# Stage 2 — Data Cleaning & Wrangling
## Google Play Store Apps · Tier A (McKinsey Standard)

`STRUCTURE.md` budgets 50–70% of project time here, and on this file it earns it. Four decisions are
made, each logged with its rationale to `data/processed/_cleaning_log.csv`:

| § | Decision | Summary |
|---|---|---|
| 2.1 | **The corrupted row** | Repair is attempted, evaluated, and **rejected** — the row is dropped, and the rejected repair is shown so the choice is auditable. |
| 2.2 | **Missingness** | Three *different mechanisms* get three *different treatments*. Nothing is imputed. |
| 2.3 | **Derived columns** | 17 new fields, including the corrected 21-bracket install ladder and the `Rating_Observed` censoring flag. |
| 2.4 | **Duplicates** | Two mechanisms are disentangled, producing **two grains** rather than one deduplicated file. |

> **The load-bearing decision is §2.4.** The data review recommends deduplicating on `App` (*"keep max
> reviews / keep latest Last Updated"*). Doing that silently deletes 85 legitimate Play Store
> cross-listings. This notebook keeps both grains and makes every downstream exhibit declare which
> one it used.

In [1]:
from __future__ import annotations

import json
import re
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd

RANDOM_STATE = 42
SCRAPE_REF = pd.Timestamp("2018-08-08")      # F7 - max(Last Updated); an assumption, flagged as one

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
RAW_CSV  = ROOT / "googleplaystore.csv" / "googleplaystore.csv"
RAW_DIR  = ROOT / "data" / "raw"
PROC_DIR = ROOT / "data" / "processed"
PROC_DIR.mkdir(parents=True, exist_ok=True)

INSTALL_LADDER = [
    "0", "0+", "1+", "5+", "10+", "50+", "100+", "500+", "1,000+", "5,000+", "10,000+",
    "50,000+", "100,000+", "500,000+", "1,000,000+", "5,000,000+", "10,000,000+",
    "50,000,000+", "100,000,000+", "500,000,000+", "1,000,000,000+",
]

pd.set_option("display.width", 170)
pd.set_option("display.max_columns", 50)

# --- the cleaning log ----------------------------------------------------------------------------
_log: list[dict] = []
def log(action, target, rows_affected, rationale, mechanism=""):
    _log.append({"step": len(_log) + 1, "action": action, "target": target,
                 "rows_affected": rows_affected, "mechanism": mechanism, "rationale": rationale})

df = pd.read_csv(RAW_CSV, dtype=str, keep_default_na=False, encoding="utf-8")
print(f"loaded {df.shape[0]:,} x {df.shape[1]} from the immutable source")
print(f"scrape reference date (assumption): {SCRAPE_REF.date()}")

loaded 10,841 x 13 from the immutable source
scrape reference date (assumption): 2018-08-08


## 2.1 The corrupted row — repair attempted, then rejected

Row 10472 has every field shifted **left by one** from `Category` onward. A right-shift repair is
*attemptable*, so it is attempted here rather than waved away — the reader should see what the repair
would actually yield before accepting the decision to drop.

In [2]:
bad = ~df["Category"].str.fullmatch(r"[A-Z_]+")
corrupt_idx = df.index[bad].tolist()
assert len(corrupt_idx) == 1, f"expected exactly 1 corrupted row, found {len(corrupt_idx)}"
i = corrupt_idx[0]

shifted = df.loc[i]
cols = list(df.columns)
# Right-shift everything from Category onward by one position; Category itself becomes unknown.
repaired = {"App": shifted["App"], "Category": None}
for src, dst in zip(cols[1:-1], cols[2:]):
    repaired[dst] = shifted[src]

comparison = pd.DataFrame({
    "as stored (shifted)": shifted,
    "if right-shift repaired": pd.Series(repaired),
})
display(comparison)

print("What the repair recovers vs. leaves unknown:")
print(f"  recovered : Rating={repaired['Rating']}  Size={repaired['Size']}  "
      f"Installs={repaired['Installs']}  Type={repaired['Type']}  "
      f"Price={repaired['Price']}  Content Rating={repaired['Content Rating']}")
print(f"  UNKNOWN   : Category (the project's primary analytical dimension)")
print(f"  UNKNOWN   : Reviews  (shifted out of the frame entirely)")

,as stored (shifted),if right-shift repaired
App,Life Made WI-Fi Touchscreen Photo Frame,Life Made WI-Fi Touchscreen Photo Frame
Category,1.9,NaN
Rating,19,1.9
Reviews,3.0M,19
Size,"1,000+",3.0M
Installs,Free,"1,000+"
Type,0,Free
Price,Everyone,0
Content Rating,,Everyone
Genres,"February 11, 2018",


What the repair recovers vs. leaves unknown:
  recovered : Rating=1.9  Size=3.0M  Installs=1,000+  Type=Free  Price=0  Content Rating=Everyone
  UNKNOWN   : Category (the project's primary analytical dimension)
  UNKNOWN   : Reviews  (shifted out of the frame entirely)


**Decision: drop the row, and record the rejected repair.**

The repair recovers six fields but leaves **`Category` unrecoverable** — and `Category` is the primary
analytical dimension of every branch in the issue tree. It also loses `Reviews` entirely. Imputing two
key fields for one row out of 10,841 changes no result; what it *would* change is the auditability of
the pipeline, since a `Category = NaN` row propagates into every group-by as a silent extra bin.

This is the STRUCTURE.md discipline: *"Never remove outliers without a documented reason."* The reason
is documented, and the rejected alternative is displayed above rather than asserted.

In [3]:
df = df.drop(index=corrupt_idx).reset_index(drop=True)
log("drop", "row 10472 (corrupted scrape artifact)", 1,
    "Left-shifted row. Right-shift repair attempted (see notebook) but leaves Category - the primary "
    "analytical dimension - and Reviews unrecoverable. Dropping 1 of 10,841 rows changes no result; "
    "retaining a Category=NaN row would add a silent bin to every group-by.",
    mechanism="scrape field dropout")
print(f"rows after drop: {len(df):,}")

# The corrupt row was also the sole source of the two stray vocabulary values the review flags.
assert (df["Type"] == "0").sum() == 0, "Type=='0' should have left with the corrupt row"
assert (df["Installs"] == "Free").sum() == 0, "Installs=='Free' should have left with the corrupt row"
print("stray vocabulary values Type=='0' and Installs=='Free' resolved with the same drop")

rows after drop: 10,840
stray vocabulary values Type=='0' and Installs=='Free' resolved with the same drop


## 2.2 Missingness — three mechanisms, three treatments

`STRUCTURE.md` requires a documented strategy *per column*, with the mechanism named (MCAR/MAR/MNAR).
The point of this section is that the three sources of missingness in this file are not the same kind
of thing, and treating them uniformly — the usual `dropna()` — destroys the most important one.

In [4]:
miss = []

# --- (1) Rating: structural censoring, MNAR by design -------------------------------------------
rating_missing = (df["Rating"] == "NaN")
miss.append({"column": "Rating", "n_missing": int(rating_missing.sum()),
             "pct": round(rating_missing.mean() * 100, 1), "mechanism": "MNAR - structural censoring",
             "treatment": "NEVER imputed; Rating_Observed flag becomes a feature"})

# --- (2) Size / Android Ver: 'Varies with device' is a real state, not an absence ----------------
size_varies = (df["Size"] == "Varies with device")
andv_varies = (df["Android Ver"] == "Varies with device")
miss.append({"column": "Size", "n_missing": int(size_varies.sum()),
             "pct": round(size_varies.mean() * 100, 1), "mechanism": "Not missing - a real state",
             "treatment": "Size_MB=NaN + Size_Varies flag (adaptive delivery)"})
miss.append({"column": "Android Ver", "n_missing": int(andv_varies.sum()),
             "pct": round(andv_varies.mean() * 100, 1), "mechanism": "Not missing - a real state",
             "treatment": "Android_Ver_Min=NaN + Android_Ver_Varies flag"})

# --- (3) genuine blanks --------------------------------------------------------------------------
for c in ["Type", "Content Rating", "Current Ver", "Android Ver"]:
    blank = (df[c].str.strip() == "")
    if blank.any():
        miss.append({"column": f"{c} (blank)", "n_missing": int(blank.sum()),
                     "pct": round(blank.mean() * 100, 2), "mechanism": "Genuine blank (MCAR)",
                     "treatment": "left NaN, counted, denominator stated"})

missingness = pd.DataFrame(miss).sort_values("n_missing", ascending=False).reset_index(drop=True)
display(missingness)

for r in missingness.itertuples():
    log("classify missingness", r.column, r.n_missing, r.treatment, mechanism=r.mechanism)

,column,n_missing,pct,mechanism,treatment
0,Size,1695,15.60,Not missing - a real state,Size_MB=NaN + Size_Varies flag (adaptive deliv...
1,Rating,1474,13.60,MNAR - structural censoring,NEVER imputed; Rating_Observed flag becomes a ...
2,Android Ver,1362,12.60,Not missing - a real state,Android_Ver_Min=NaN + Android_Ver_Varies flag
3,Current Ver (blank),1,0.01,Genuine blank (MCAR),"left NaN, counted, denominator stated"


**Why `Rating` is never imputed.** Stage 1 proved the missingness is generated by the outcome:
zero-review apps are 100% unrated and the rate falls monotonically to 0% at scale. Imputing it — with
a mean, a median, or a model — would **fabricate the long tail**, which is precisely the population
the analysis is about. The flag is the information; the value is not recoverable.

**Why `"Varies with device"` is not missingness.** It is a publisher choice to ship adaptive builds.
Treating it as absent would drop 15.6% of the catalog non-randomly (it correlates with publisher
scale). It becomes a flag, and Stage 3 tests whether the flag itself carries signal.

**Model-time imputation happens inside the `Pipeline`** (Stage 4), fit on training folds only, with an
explicit missing-indicator. Nothing is imputed at rest.

## 2.3 Derived columns

Implements `DOCS/googleplaystore_dataset_review.md` §7, with the corrections established in the
implementation plan's pre-flight profiling.

In [5]:
out = df.copy()

# --- numeric casts --------------------------------------------------------------------------------
out["Reviews_int"] = pd.to_numeric(out["Reviews"], errors="raise").astype("int64")
out["Rating_num"] = pd.to_numeric(out["Rating"].replace("NaN", np.nan), errors="coerce")
out["Rating_Observed"] = out["Rating_num"].notna()            # <- the F1 censoring feature

# --- install ladder: ordinal code, floor, macro-tier, log ------------------------------------------
out["Installs_tier"] = pd.Categorical(out["Installs"], categories=INSTALL_LADDER, ordered=True).codes
assert (out["Installs_tier"] >= 0).all(), "an Installs value fell outside the 21-bracket ladder"
out["Installs_floor"] = out["Installs"].str.replace(r"[+,]", "", regex=True).astype("int64")
out["log_installs"] = np.log10(out["Installs_floor"].clip(lower=1))

MACRO_BINS = [-1, 1_000, 100_000, 10_000_000, np.inf]
MACRO_LABELS = ["Niche (<1k)", "Emerging (1k-100k)", "Established (100k-10M)", "Blockbuster (10M+)"]
out["Installs_macro"] = pd.cut(out["Installs_floor"], bins=MACRO_BINS, labels=MACRO_LABELS)

# --- size ------------------------------------------------------------------------------------------
def _size_mb(s: str) -> float:
    if s.endswith("M"):
        return float(s[:-1])
    if s.endswith("k"):
        return float(s[:-1]) / 1024
    return np.nan
out["Size_MB"] = out["Size"].map(_size_mb)
out["Size_Varies"] = (out["Size"] == "Varies with device")
assert out.loc[~out["Size_Varies"], "Size_MB"].notna().all(), "a non-'Varies' Size failed to parse"

# --- price ------------------------------------------------------------------------------------------
out["Price_numeric"] = pd.to_numeric(out["Price"].str.replace("$", "", regex=False), errors="raise")
out["Is_Paid"] = out["Type"].map({"Paid": True, "Free": False})       # blank -> NaN, kept as such
out["Is_Novelty_Price"] = out["Price_numeric"] >= 79.99               # F8 - flag, never a deletion

# --- dates / recency -----------------------------------------------------------------------------
out["Last_Updated_date"] = pd.to_datetime(out["Last Updated"], format="%B %d, %Y", errors="raise")
out["Days_Since_Update"] = (SCRAPE_REF - out["Last_Updated_date"]).dt.days

# --- genres ----------------------------------------------------------------------------------------
out["Primary_Genre"] = out["Genres"].str.split(";").str[0]
out["Genre_Count"] = out["Genres"].str.count(";") + 1

# --- android version ---------------------------------------------------------------------------------
out["Android_Ver_Varies"] = (out["Android Ver"] == "Varies with device")
out["Android_Ver_Min"] = pd.to_numeric(
    out["Android Ver"].str.extract(r"^(\d+(?:\.\d+)?)")[0], errors="coerce")

for c in ["Reviews_int", "Rating_Observed", "Installs_tier", "Installs_floor", "log_installs",
          "Installs_macro", "Size_MB", "Size_Varies", "Price_numeric", "Is_Paid",
          "Is_Novelty_Price", "Last_Updated_date", "Days_Since_Update", "Primary_Genre",
          "Genre_Count", "Android_Ver_Varies", "Android_Ver_Min"]:
    log("derive", c, len(out), "per DOCS review section 7 + implementation-plan corrections")

print(f"{out.shape[1] - df.shape[1]} derived columns added -> {out.shape[1]} total\n")
out[["App", "Category", "Rating_num", "Rating_Observed", "Reviews_int", "Installs_tier",
     "Installs_macro", "Size_MB", "Price_numeric", "Days_Since_Update"]].head(4)

18 derived columns added -> 31 total



,App,Category,Rating_num,Rating_Observed,Reviews_int,Installs_tier,Installs_macro,Size_MB,Price_numeric,Days_Since_Update
0,Photo Editor & Candy Camera & Grid & ScrapBook,ART_AND_DESIGN,4.1,True,159,10,Emerging (1k-100k),19.0,0.0,213
1,Coloring book moana,ART_AND_DESIGN,3.9,True,967,13,Established (100k-10M),14.0,0.0,205
2,"U Launcher Lite – FREE Live Cool Themes, Hide ...",ART_AND_DESIGN,4.7,True,87510,15,Established (100k-10M),8.7,0.0,7
3,Sketch - Draw & Paint,ART_AND_DESIGN,4.5,True,215644,17,Blockbuster (10M+),25.0,0.0,61


### Two validations, one of which is a *passed* check worth reporting

The data review suggests cross-checking `Is_Paid` against `Price > 0`. It passes with **zero
mismatches** — reported here because a passed consistency check is evidence about data quality, and
silently omitting it would leave the reader unable to tell whether it was run.

### One column deliberately *not* created

The review proposes `Reviews_per_Install` as an engagement proxy. It is **not created**. Its numerator
is the leakage term (`Reviews` reproduces r² = 0.93 of installs) and its denominator is a bucket
floor — a ratio of a leak to a lower bound. It would be banned from every model anyway, and creating
it invites accidental use.

In [6]:
checks = []
mismatch = out[out["Is_Paid"].notna() & (out["Is_Paid"] != (out["Price_numeric"] > 0))]
checks.append({"check": "Is_Paid agrees with Price_numeric > 0",
               "result": f"{len(mismatch)} mismatches in {int(out['Is_Paid'].notna().sum()):,} rows",
               "status": "PASS" if len(mismatch) == 0 else "FAIL"})

checks.append({"check": "Price_numeric >= 0", "result": f"min = ${out['Price_numeric'].min():.2f}",
               "status": "PASS" if (out["Price_numeric"] >= 0).all() else "FAIL"})
checks.append({"check": "Rating_num within [1.0, 5.0] where observed",
               "result": f"range {out['Rating_num'].min():.1f} - {out['Rating_num'].max():.1f}",
               "status": "PASS" if out["Rating_num"].dropna().between(1, 5).all() else "FAIL"})
checks.append({"check": "Last_Updated_date <= scrape reference (no future dates)",
               "result": f"max = {out['Last_Updated_date'].max().date()}",
               "status": "PASS" if (out["Last_Updated_date"] <= SCRAPE_REF).all() else "FAIL"})
checks.append({"check": "Reviews_int >= 0", "result": f"min = {out['Reviews_int'].min():,}",
               "status": "PASS" if (out["Reviews_int"] >= 0).all() else "FAIL"})
checks.append({"check": "Installs_macro covers every row",
               "result": f"{int(out['Installs_macro'].isna().sum())} unassigned",
               "status": "PASS" if out["Installs_macro"].notna().all() else "FAIL"})

business_rules = pd.DataFrame(checks)
display(business_rules)
assert (business_rules["status"] == "PASS").all(), "a business rule failed"
log("validate", "business rules", len(out),
    "6 rules incl. the review's suggested Is_Paid/Price cross-check - 0 mismatches (a PASSED check)")

n_novelty_rows = int(out["Is_Novelty_Price"].sum())
n_novelty_apps = int(out.loc[out["Is_Novelty_Price"], "App"].nunique())
print(f"\nNovelty-price listings flagged (>= $79.99): {n_novelty_rows} rows / "
      f"{n_novelty_apps} distinct apps - flagged, NOT deleted")
print(out.loc[out['Is_Novelty_Price'], 'Category'].value_counts().to_string())
print(f"\nEffect on paid-price statistics (row grain):")
paid = out[out["Price_numeric"] > 0]
print(f"  with novelty apps    : mean ${paid['Price_numeric'].mean():6.2f}   "
      f"median ${paid['Price_numeric'].median():.2f}")
pn = paid[~paid["Is_Novelty_Price"]]
print(f"  without              : mean ${pn['Price_numeric'].mean():6.2f}   "
      f"median ${pn['Price_numeric'].median():.2f}")
print(f"  -> {n_novelty_apps} apps move the mean by "
      f"${paid['Price_numeric'].mean() - pn['Price_numeric'].mean():.2f} and the median by $0.00;")
print("     every price statistic downstream is therefore a MEDIAN")

,check,result,status
0,Is_Paid agrees with Price_numeric > 0,"0 mismatches in 10,839 rows",PASS
1,Price_numeric >= 0,min = $0.00,PASS
2,"Rating_num within [1.0, 5.0] where observed",range 1.0 - 5.0,PASS
3,Last_Updated_date <= scrape reference (no futu...,max = 2018-08-08,PASS
4,Reviews_int >= 0,min = 0,PASS
5,Installs_macro covers every row,0 unassigned,PASS



Novelty-price listings flagged (>= $79.99): 23 rows / 22 distinct apps - flagged, NOT deleted
Category
FINANCE         7
LIFESTYLE       6
FAMILY          4
MEDICAL         3
EVENTS          1
PRODUCTIVITY    1
BUSINESS        1

Effect on paid-price statistics (row grain):
  with novelty apps    : mean $ 13.92   median $2.99
  without              : mean $  4.84   median $2.99
  -> 22 apps move the mean by $9.09 and the median by $0.00;
     every price statistic downstream is therefore a MEDIAN


## 2.4 Duplicates — two mechanisms, therefore two grains

This is the decision the rest of the project depends on.

The data review reports *"1,181 duplicate `App` rows … suggesting re-scrapes … Needs a dedup decision
(keep max reviews / keep latest `Last Updated`)."* Repeated app names have **two unrelated causes**,
and only one of them is a defect:

| Mechanism | What it is | Correct treatment |
|---|---|---|
| **Re-scrape repetition** | The same listing captured more than once; rows differ only in `Reviews` (the counter ticked between captures) | collapse — keep max `Reviews` |
| **Play Store cross-listing** | An app genuinely listed under **two real categories**. `FAMILY` is an *overlay* category, not a peer of `GAME` — an app can hold both | **keep both rows** |

Collapsing on `App` alone treats the second as if it were the first.

In [7]:
# --- level 1: byte-identical rows ------------------------------------------------------------------
raw_cols = list(df.columns)
n_exact = int(out.duplicated(subset=raw_cols).sum())

# --- level 2: repeated app names, and what varies within them ---------------------------------------
rep = (out.groupby("App")
          .agg(n=("App", "size"), n_cat=("Category", "nunique"),
               n_rev=("Reviews_int", "nunique"), n_inst=("Installs", "nunique")))
repeated = rep[rep["n"] > 1]

# --- level 3: genuine cross-listings ------------------------------------------------------------------
cross = rep[rep["n_cat"] > 1]
out["Is_Cross_Listed"] = out["App"].isin(cross.index)

print(f"level 1  byte-identical rows (all 13 raw cols) : {n_exact:,}")
print(f"level 2  app names appearing more than once    : {len(repeated):,}"
      f"  (max repeats: {int(rep['n'].max())})")
print(f"         ... of which differ only in Reviews   : {int((repeated['n_rev'] > 1).sum()):,}"
      f"  ({(repeated['n_rev'] > 1).mean()*100:.1f}%)")
print(f"level 3  apps listed under >1 real Category    : {len(cross):,}  <- NOT duplicates\n")

pairs = (out[out["Is_Cross_Listed"]][["App", "Category"]].drop_duplicates()
         .groupby("App")["Category"].apply(lambda s: " | ".join(sorted(s))))
print("the cross-listing pairs:")
print(pairs.value_counts().to_string())

level 1  byte-identical rows (all 13 raw cols) : 483
level 2  app names appearing more than once    : 798  (max repeats: 9)
         ... of which differ only in Reviews   : 517  (64.8%)
level 3  apps listed under >1 real Category    : 85  <- NOT duplicates

the cross-listing pairs:
Category
FAMILY | GAME                   34
ENTERTAINMENT | FAMILY          23
EDUCATION | FAMILY              17
ART_AND_DESIGN | FAMILY          3
GAME | SPORTS                    2
FAMILY | VIDEO_PLAYERS           2
DATING | TOOLS                   1
BUSINESS | COMMUNICATION         1
HEALTH_AND_FITNESS | MEDICAL     1
HOUSE_AND_HOME | TOOLS           1


In [8]:
# Worked example: one app that is BOTH re-scraped and cross-listed.
ex = out[out["App"] == "8 Ball Pool"][["App", "Category", "Rating_num", "Reviews_int",
                                       "Installs", "Last Updated"]]
print("'8 Ball Pool' - 6 GAME captures with a ticking review counter, plus a real SPORTS listing:")
display(ex.sort_values(["Category", "Reviews_int"]))
print("Collapsing on App alone keeps ONE of these rows and silently deletes the SPORTS listing.")

'8 Ball Pool' - 6 GAME captures with a ticking review counter, plus a real SPORTS listing:


,App,Category,Rating_num,Reviews_int,Installs,Last Updated
1675,8 Ball Pool,GAME,4.5,14198297,"100,000,000+","July 31, 2018"
1703,8 Ball Pool,GAME,4.5,14198602,"100,000,000+","July 31, 2018"
1755,8 Ball Pool,GAME,4.5,14200344,"100,000,000+","July 31, 2018"
1844,8 Ball Pool,GAME,4.5,14200550,"100,000,000+","July 31, 2018"
1970,8 Ball Pool,GAME,4.5,14201604,"100,000,000+","July 31, 2018"
1871,8 Ball Pool,GAME,4.5,14201891,"100,000,000+","July 31, 2018"
3953,8 Ball Pool,SPORTS,4.5,14184910,"100,000,000+","July 31, 2018"


Collapsing on App alone keeps ONE of these rows and silently deletes the SPORTS listing.


### The dual-grain resolution

| Grain | Rule | Rows | Used for |
|---|---|---|---|
| **Row-level** | corrupt row dropped only | 10,840 | the forensic base — evidence for the duplicate analysis itself |
| **`(App, Category)`** | max `Reviews` within each pair | **9,744** | **market-structure statistics** — an app in two categories votes in both, which is true |
| **`App`** | max `Reviews` across all rows | **9,659** | **modelling** — one row per app, so no near-duplicate straddles a CV fold |

Every downstream exhibit prints which grain and which denominator it used.

In [9]:
srt = out.sort_values("Reviews_int")

apps_clean      = out.copy()                                                    # 10,840 - forensic
apps_by_cat     = srt.drop_duplicates(["App", "Category"], keep="last").copy()  #  9,744 - structure
apps_app_grain  = srt.drop_duplicates(["App"], keep="last").copy()              #  9,659 - modelling

for name, frame in [("apps_clean", apps_clean), ("apps_by_category", apps_by_cat),
                    ("apps_app_grain", apps_app_grain)]:
    print(f"{name:18s} {len(frame):>6,} rows")

assert len(apps_by_cat) == 9_744, f"category grain is {len(apps_by_cat):,}, expected 9,744"
assert len(apps_app_grain) == 9_659, f"app grain is {len(apps_app_grain):,}, expected 9,659"

lost = len(apps_by_cat) - len(apps_app_grain)
print(f"\nrows the App grain gives up vs the (App, Category) grain: {lost}"
      f"  = the {len(cross)} cross-listed apps' extra listings")

log("dedup", "byte-identical rows", n_exact, "exact re-scrape repetition; collapsed by max Reviews",
    mechanism="re-scrape repetition")
log("dedup", "(App, Category) grain -> apps_by_category.parquet", len(out) - len(apps_by_cat),
    "market-structure base: keeps genuine cross-listings so a 2-category app votes in both",
    mechanism="re-scrape repetition")
log("dedup", "App grain -> apps_app_grain.parquet", len(out) - len(apps_app_grain),
    f"modelling base: one row per app so no near-duplicate straddles a CV fold; gives up {lost} "
    "legitimate cross-listings, which is why the category grain exists separately",
    mechanism="re-scrape repetition + cross-listing")

apps_clean         10,840 rows
apps_by_category    9,744 rows
apps_app_grain      9,659 rows

rows the App grain gives up vs the (App, Category) grain: 85  = the 85 cross-listed apps' extra listings


### What the review's recommendation would have cost

Deduplicating on `App` alone — the review's suggestion — is the *modelling* grain, and it is correct
**for modelling**. Used for *market structure*, it under-counts every category that participates in a
cross-listing. `FAMILY`, the largest category in the catalog, takes the biggest hit.

In [10]:
delta = pd.DataFrame({
    "(App, Category) grain": apps_by_cat["Category"].value_counts(),
    "App grain": apps_app_grain["Category"].value_counts(),
})
delta["lost"] = delta["(App, Category) grain"] - delta["App grain"]
affected = delta[delta["lost"] > 0].sort_values("lost", ascending=False)
print(f"categories under-counted by App-only dedup: {len(affected)} of {out['Category'].nunique()}\n")
display(affected)
print(f"total listings lost: {int(affected['lost'].sum())}")

categories under-counted by App-only dedup: 10 of 33



,"(App, Category) grain",App grain,lost
Category,,,
FAMILY,1909,1875,34
ENTERTAINMENT,102,86,16
GAME,960,946,14
EDUCATION,119,107,12
ART_AND_DESIGN,64,61,3
SPORTS,327,325,2
COMMUNICATION,316,315,1
DATING,171,170,1
HOUSE_AND_HOME,74,73,1


total listings lost: 85


### Three explicit non-actions

1. **The 20 novelty-price apps are not removed.** They are real listings and a documented catalog
   phenomenon. Deleting them would be undocumented outlier removal — a STRUCTURE.md anti-pattern.
   They are flagged, and every price statistic downstream is a **median** with the flag's effect shown.
2. **No fuzzy app-name merging.** 9,659 distinct names survive exact dedup; fuzzy matching on real
   product names would create false merges to fix a problem not yet demonstrated. If Stage 3 surfaces
   genuine variants, this loops back — logged.
3. **The 596 zero-review apps are not dropped.** They are the censored tail and the entire subject of
   Key Line 1. Dropping them is the single most common error made on this dataset.

In [11]:
for a, t, n, r in [
    ("retain", "novelty-price listings (>= $79.99)", int(out["Is_Novelty_Price"].sum()),
     "Real listings, documented phenomenon. Flagged not deleted; all price stats are medians."),
    ("no-action", "fuzzy app-name matching", 0,
     "9,659 distinct names after exact dedup; fuzzy merging would create false merges to fix an "
     "undemonstrated problem. Loops back if Stage 3 surfaces genuine variants."),
    ("retain", "zero-review apps", int((out["Reviews_int"] == 0).sum()),
     "The censored tail (100% unrated by construction) and the subject of Key Line 1. "
     "Dropping them is the most common error made on this dataset."),
]:
    log(a, t, n, r)
print("three non-actions logged")

three non-actions logged


## 2.5 Output & cleaning log

In [12]:
KEEP = [
    "App", "Category", "Rating", "Reviews", "Size", "Installs", "Type", "Price", "Content Rating",
    "Genres", "Last Updated", "Current Ver", "Android Ver",
    "Rating_num", "Rating_Observed", "Reviews_int", "Installs_tier", "Installs_floor",
    "log_installs", "Installs_macro", "Size_MB", "Size_Varies", "Price_numeric", "Is_Paid",
    "Is_Novelty_Price", "Last_Updated_date", "Days_Since_Update", "Primary_Genre", "Genre_Count",
    "Android_Ver_Varies", "Android_Ver_Min", "Is_Cross_Listed",
]
for name, frame in [("apps_clean", apps_clean), ("apps_by_category", apps_by_cat),
                    ("apps_app_grain", apps_app_grain)]:
    frame[KEEP].reset_index(drop=True).to_parquet(PROC_DIR / f"{name}.parquet", index=False)

cleaning_log = pd.DataFrame(_log)
cleaning_log.to_csv(PROC_DIR / "_cleaning_log.csv", index=False)
missingness.to_csv(PROC_DIR / "_missingness.csv", index=False)

print("written to data/processed/:")
for f in sorted(PROC_DIR.iterdir()):
    print(f"  {f.name:28s} {f.stat().st_size:>9,} bytes")
print(f"\ncleaning log: {len(cleaning_log)} actions")
display(cleaning_log[["step", "action", "target", "rows_affected", "mechanism"]])

written to data/processed/:
  _cleaning_log.csv                3,586 bytes
  _missingness.csv                   421 bytes
  apps_app_grain.parquet         504,782 bytes
  apps_by_category.parquet       507,962 bytes
  apps_clean.parquet             534,120 bytes

cleaning log: 29 actions


,step,action,target,rows_affected,mechanism
0,1,drop,row 10472 (corrupted scrape artifact),1,scrape field dropout
1,2,classify missingness,Size,1695,Not missing - a real state
2,3,classify missingness,Rating,1474,MNAR - structural censoring
3,4,classify missingness,Android Ver,1362,Not missing - a real state
4,5,classify missingness,Current Ver (blank),1,Genuine blank (MCAR)
5,6,derive,Reviews_int,10840,
6,7,derive,Rating_Observed,10840,
7,8,derive,Installs_tier,10840,
8,9,derive,Installs_floor,10840,
9,10,derive,log_installs,10840,


In [13]:
# --- dtype contract, asserted on the file that ships forward -------------------------------------
chk = pd.read_parquet(PROC_DIR / "apps_app_grain.parquet")
contract = {
    "Reviews_int": "int64", "Installs_tier": "int8", "Installs_floor": "int64",
    "log_installs": "float64", "Size_MB": "float64", "Price_numeric": "float64",
    "Days_Since_Update": "int64", "Rating_num": "float64", "Genre_Count": "int64",
}
report = []
for col, want in contract.items():
    got = str(chk[col].dtype)
    report.append({"column": col, "expected": want, "actual": got,
                   "status": "PASS" if got == want else "CHECK"})
dtypes = pd.DataFrame(report)
display(dtypes)

print(f"Rating_Observed  : {chk['Rating_Observed'].dtype}  "
      f"({chk['Rating_Observed'].sum():,} observed / {len(chk):,})")
print(f"Installs_macro   : {chk['Installs_macro'].dtype}")
print(f"Last_Updated_date: {chk['Last_Updated_date'].dtype}")
print(f"\nround-trip OK: {len(chk):,} rows x {chk.shape[1]} columns")

,column,expected,actual,status
0,Reviews_int,int64,int64,PASS
1,Installs_tier,int8,int8,PASS
2,Installs_floor,int64,int64,PASS
3,log_installs,float64,float64,PASS
4,Size_MB,float64,float64,PASS
5,Price_numeric,float64,float64,PASS
6,Days_Since_Update,int64,int64,PASS
7,Rating_num,float64,float64,PASS
8,Genre_Count,int64,int64,PASS


Rating_Observed  : bool  (8,196 observed / 9,659)
Installs_macro   : category
Last_Updated_date: datetime64[us]

round-trip OK: 9,659 rows x 32 columns


## Stage 2 — Gate Checklist

- [x] **Missing value strategy documented per column** — `data/processed/_missingness.csv`, with the
      *mechanism* named for each: `Rating` is **MNAR structural censoring** (never imputed),
      `"Varies with device"` is **a real state, not an absence** (flagged), and the handful of blanks
      are genuine (counted, denominators stated). Nothing is imputed at rest.
- [x] **Duplicates assessed and handled with justification** — three levels distinguished
      (byte-identical, re-scrape repetition, genuine cross-listing) producing **two grains** rather
      than one. The review's `App`-only recommendation is shown to under-count **10 categories by 85
      listings**, `FAMILY` worst.
- [x] **All columns have correct dtypes** — contract asserted on the round-tripped parquet.
- [x] **Business-rule validations pass** — 6 rules, all PASS, including the review's suggested
      `Is_Paid` × `Price > 0` cross-check (**0 mismatches** — a passed check, reported as such).
- [x] **Cleaning log exists** — `_cleaning_log.csv`: every action with rows affected, mechanism, and
      rationale, including the three deliberate **non-actions**.
- [x] **Cleaned data saved to `data/processed/`** — three grains, parquet.

**Next:** `03_eda.ipynb` — hypothesis-driven EDA, one section per issue-tree branch, action titles
throughout.